In [52]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import tensorflow as tf
import kagglehub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# New Section

In [53]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("preetviradiya/brian-tumor-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brian-tumor-dataset' dataset.
Path to dataset files: /kaggle/input/brian-tumor-dataset


In [54]:
print(os.listdir(path))

['metadata_rgb_only.csv', 'Brain Tumor Data Set', 'metadata.csv']


In [55]:
images_path=os.path.join(path,'Brain Tumor Data Set',"Brain Tumor Data Set", "Brain Tumor")
print(images_path)

/kaggle/input/brian-tumor-dataset/Brain Tumor Data Set/Brain Tumor Data Set/Brain Tumor


In [56]:
print(len(os.listdir(images_path)))
print(os.listdir(images_path)[:5])

2513
['Cancer (489).jpg', 'Cancer (72).tif', 'Cancer (2411).jpg', 'Cancer (1065).jpg', 'Cancer (1014).jpg']


In [57]:
images_path1=os.path.join(path,'Brain Tumor Data Set',"Brain Tumor Data Set", "Healthy")
print(images_path)

/kaggle/input/brian-tumor-dataset/Brain Tumor Data Set/Brain Tumor Data Set/Brain Tumor


In [58]:
print(len(os.listdir(images_path1)))
print(os.listdir(images_path1)[:5])

2087
['Not Cancer  (446).jpg', 'Not Cancer  (1646).jpg', 'Not Cancer  (1548).jpg', 'Not Cancer  (1864).jpg', 'Not Cancer  (1736).jpg']


In [59]:
import os
from PIL import Image

# ✅ Correct source paths
tumor_src = os.path.join(path, "Brain Tumor Data Set", "Brain Tumor Data Set", "Brain Tumor")
healthy_src = os.path.join(path, "Brain Tumor Data Set", "Brain Tumor Data Set", "Healthy")

# ✅ Target paths
base_dir = "/kaggle/working/brain_dataset"
tumor_dir = os.path.join(base_dir, "tumor")
healthy_dir = os.path.join(base_dir, "healthy")

os.makedirs(tumor_dir, exist_ok=True)
os.makedirs(healthy_dir, exist_ok=True)

valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif")

# 🔁 Function to clean + copy
def process_images(src, target):
    for file in os.listdir(src):

        file_path = os.path.join(src, file)

        if not file.lower().endswith(valid_ext):
            continue

        try:
            img = Image.open(file_path).convert("RGB")

            new_name = file.split(".")[0] + ".jpg"
            save_path = os.path.join(target, new_name)

            img.save(save_path, "JPEG")

        except:
            print("Skipped:", file)

# 🚀 Run for both classes
process_images(tumor_src, tumor_dir)
process_images(healthy_src, healthy_dir)

print("✅ Dataset ready")

✅ Dataset ready


In [ ]:
print("Tumor:", len(os.listdir(tumor_dir)))
print("Healthy:", len(os.listdir(healthy_dir)))

In [ ]:
import tensorflow as tf

def load_data(base_dir):

    train_ds = tf.keras.utils.image_dataset_from_directory(
        base_dir,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=(224,224),
        batch_size=32
    )


    val_ds = tf.keras.utils.image_dataset_from_directory(
        base_dir,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=(224,224),
        batch_size=32
    )

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# your dataset path
base_dir = "/kaggle/working/brain_dataset"

tumor_dir = os.path.join(base_dir, "tumor")
healthy_dir = os.path.join(base_dir, "healthy")

# count images
data = {
    "Class": ["Tumor", "Healthy"],
    "Count": [
        len(os.listdir(tumor_dir)),
        len(os.listdir(healthy_dir))
    ]
}

df = pd.DataFrame(data)

# plot (same style as your code)
plt.figure()
df.set_index("Class")["Count"].plot(kind='bar')

plt.title("Brain Tumor vs Healthy Image Count")
plt.xlabel("Class")
plt.ylabel("Number of Images")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

class_names = train_ds.class_names   # ['healthy', 'tumor']

plt.figure(figsize=(10,10))

for images, labels in train_ds.take(1):

    for i in range(25):

        plt.subplot(5,5,i+1)

        plt.xticks([])
        plt.yticks([])
        plt.grid(False)

        plt.imshow(images[i].numpy().astype("uint8"))

        # ✅ FIX (important)
        plt.xlabel(class_names[labels[i].numpy()])

plt.show()

In [ ]:
    # Normalize
    train_ds = train_ds.map(lambda x, y: (x/255.0, y))
    val_ds   = val_ds.map(lambda x, y: (x/255.0, y))

    return train_ds, val_ds

In [ ]:
import tensorflow as tf

data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),   # small rotation
    tf.keras.layers.RandomZoom(0.05),       # slight zoom
    tf.keras.layers.RandomContrast(0.1)     # improve robustness
])

In [ ]:
from PIL import Image
import os

sizes = []

for folder in ["tumor", "healthy"]:
    folder_path = os.path.join(base_dir, folder)

    for file in os.listdir(folder_path)[:50]:
        img = Image.open(os.path.join(folder_path, file))
        sizes.append(img.size)

print("Sample image sizes:", sizes[:10])